In [1]:
import cv2

# Charger la vidéo
cap = cv2.VideoCapture('map.mp4')

# Initialiser un compteur pour le nombre total de petits cercles noirs
total_small_circles = 0

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convertir en niveaux de gris
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)

    # Appliquer un seuil pour isoler les points noirs
    _, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY_INV)

    # Détecter les contours des points noirs
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

    # Filtrer et compter les petits cercles noirs
    for contour in contours:
        # Approximer le contour à un cercle
        perimeter = cv2.arcLength(contour, True)
        approx = cv2.approxPolyDP(contour, 0.04 * perimeter, True)

        # Filtrer les contours en fonction du nombre de sommets et de la taille
        if len(approx) > 8:  # Assure que la forme est proche d'un cercle
            area = cv2.contourArea(contour)
            if 10 < area < 1000:  # Ajuster ces valeurs selon la taille des cercles que vous recherchez
                total_small_circles += 1

    # Afficher la frame avec les contours détectés (optionnel)
    cv2.drawContours(frame, contours, -1, (0, 255, 0), 2)
    cv2.imshow('Frame', frame)

    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

# Libérer les ressources
cap.release()
cv2.destroyAllWindows()

# Imprimer le nombre total de petits cercles noirs détectés
print(f'Nombre total de petits cercles noirs dans la vidéo : {total_small_circles}')


Nombre total de petits cercles noirs dans la vidéo : 0


In [2]:
pip install cv2


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


ERROR: Could not find a version that satisfies the requirement cv2 (from versions: none)
ERROR: No matching distribution found for cv2


In [4]:
!pip install opencv-python



Defaulting to user installation because normal site-packages is not writeable
     ---------------------------------------- 38.8/38.8 MB 1.5 MB/s eta 0:00:00


In [16]:
import cv2
import numpy as np

# Charger l'image de la carte
image = cv2.imread('map.png')

# Convertir en niveaux de gris
gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

# Appliquer un seuil pour isoler les points noirs
_, thresh = cv2.threshold(gray, 50, 255, cv2.THRESH_BINARY_INV)

# Appliquer une opération morphologique pour supprimer le bruit
kernel = np.ones((3,3), np.uint8)
thresh = cv2.erode(thresh, kernel, iterations=2)
thresh = cv2.dilate(thresh, kernel, iterations=2)

# Détecter les contours des points noirs
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Initialiser un compteur pour les points noirs
small_black_points = 0

# Filtrer les contours pour ne garder que les petits cercles noirs
for contour in contours:
    area = cv2.contourArea(contour)
    perimeter = cv2.arcLength(contour, True)
    circularity = 4 * np.pi * (area / (perimeter * perimeter))

    if 0.7 < circularity <= 1.2 and 10 < area < 200:  # Ajustez ces valeurs si nécessaire
        small_black_points += 1
        # Dessiner les contours détectés (optionnel)
        cv2.drawContours(image, [contour], -1, (0, 255, 0), 2)

# Afficher le nombre total de petits points noirs détectés
print(f'Nombre total de petits points noirs détectés : {small_black_points}')

# Afficher l'image avec les contours détectés
cv2.imshow('Contours détectés', image)
cv2.waitKey(0)
cv2.destroyAllWindows()


Nombre total de petits points noirs détectés : 48


In [11]:
import cv2
import numpy as np

# Read the image
img = cv2.imread('map3.png', cv2.IMREAD_GRAYSCALE)

# Thresholding to create a binary image
thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV)[1]

# Find contours
contours, hierarchy = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Count the number of contours
print("Number of black dots:", len(contours))

# Display the image with contours (optional)
cv2.imshow('Image', thresh)
cv2.waitKey(0)
cv2.destroyAllWindows()


Number of black dots: 0


In [8]:
import cv2
import numpy as np
from sklearn.cluster import DBSCAN

def detect_black_dots(frame):
    """Detect black dots in a frame."""
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    thresh = cv2.threshold(gray, 10, 255, cv2.THRESH_BINARY_INV)[1]  # Ajuster le seuil si nécessaire
    contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    points = []
    for contour in contours:
        if cv2.contourArea(contour) > 10:  # Filtrer les petits contours
            for point in contour:
                x, y = point[0]
                points.append((x, y))
    return points, thresh

def cluster_points(points, eps=30, min_samples=5):
    """Cluster points using DBSCAN."""
    if len(points) == 0:
        return []
    dbscan = DBSCAN(eps=eps, min_samples=min_samples).fit(points)
    labels = dbscan.labels_
    unique_labels = set(labels)
    clusters = [np.mean(np.array(points)[labels == label], axis=0) for label in unique_labels if label != -1]
    return clusters

# Ouvrir la vidéo
cap = cv2.VideoCapture('map3.mp4')

all_points = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    points, thresh = detect_black_dots(frame)
    all_points.extend(points)

    # Afficher les images (optionnel)
    cv2.imshow('Threshold Image', thresh)
    cv2.imshow('Original Frame', frame)

    # Appuyer sur 'q' pour quitter
    if cv2.waitKey(1) & 0xFF == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()

# Convertir en tableau numpy et regrouper les points
points_array = np.array(all_points)
clusters = cluster_points(points_array)

# Afficher le nombre total de points uniques détectés
print(f"Total number of unique black dots: {len(clusters)}")


Total number of unique black dots: 0


In [5]:
import cv2
import numpy as np
import pyodbc

# Step 1: Detect black dots using OpenCV
# Read the image
img = cv2.imread('map3.png', cv2.IMREAD_GRAYSCALE)

# Thresholding to create a binary image
thresh = cv2.threshold(img, 0, 255, cv2.THRESH_BINARY_INV)[1]

# Find contours
contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

# Count the number of contours (black dots)
number_of_black_dots = len(contours)
print("Number of black dots:", number_of_black_dots)

# Step 2: SQL Server connection details
server = 'LAPTOP-HED7JE2F'
database = 'PGS_STAGE_DW'
username = 'sa'
password = 'sa@123@123'

# Establish connection
conn = pyodbc.connect(f'DRIVER={{SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}')
cursor = conn.cursor()

# Step 3: Check if table exists, and create if it does not
create_table_query = '''
IF NOT EXISTS (
    SELECT * FROM INFORMATION_SCHEMA.TABLES 
    WHERE TABLE_NAME = 'nb_stations'
)
BEGIN
    CREATE TABLE nb_stations (
        nom_concurent NVARCHAR(50),
        nombres INT
    )
END
'''
cursor.execute(create_table_query)
print("Table nb_stations checked/created successfully.")

# Step 4: Insert data into `nb_stations`
insert_query = '''
INSERT INTO nb_stations (nom_concurent, nombres)
VALUES (?, ?)
'''
cursor.execute(insert_query, ('Total_energies', number_of_black_dots))

# Commit the transaction and close the connection
conn.commit()
conn.close()

print("Data successfully inserted into nb_stations.")


Number of black dots: 117
Table nb_stations checked/created successfully.
Data successfully inserted into nb_stations.


In [ ]:
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
import time
import re

# Spécifiez le chemin du ChromeDriver via la classe Service
service = Service('C:/Users/user/Desktop/stage/chromedriver-win64/chromedriver.exe')
driver = webdriver.Chrome(service=service)

# Accéder à la page web
driver.get("https://olaenergy.com/service-stations/")

# Attendre le chargement complet de la page
time.sleep(5)  # Ajustez selon le temps nécessaire

# Sélectionner l'option "Tunisia"
select_element = driver.find_element(By.NAME, 'place_country')
select = Select(select_element)
select.select_by_visible_text('Tunisia')

# Attendre que la page se mette à jour après la sélection
time.sleep(5)  # Ajustez selon le temps nécessaire

# Sélectionner l'option "Service Station"
select_category = driver.find_element(By.NAME, 'place_category')
select = Select(select_category)
select.select_by_visible_text('service station')

# Attendre que la page se mette à jour après la sélection
time.sleep(5)  # Ajustez selon le temps nécessaire

# Extraire le code source HTML de la page
page_source = driver.page_source

# Rechercher toutes les occurrences de 'generic.png' dans le code source HTML
pattern = re.compile(r'Tunisia')
matches = pattern.findall(page_source)

# Compter le nombre d'occurrences
count = len(matches)

print(f'Number of occurrences of "generic.png": {count}')

# Fermer le navigateur
driver.quit()


In [30]:
import cv2
import numpy as np

total_pins = 0

for i in range(len(canvases)):
    # Charger chaque image
    image = cv2.imread(f'canvas_{i}.png')

    # Convertir en niveaux de gris
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Appliquer un filtre pour détecter les contours
    edges = cv2.Canny(gray, 150, 150)

    # Trouver les contours
    contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # Compter le nombre de contours détectés
    total_pins += len(contours)

print(f'Nombre total de stations-service (approximatif) : {total_pins}')


Nombre total de stations-service (approximatif) : 648


In [ ]:
import cv2
import numpy as np

total_pins = 0

for i in range(len(canvases)):
    # Charger chaque image
    image = cv2.imread(f'canvas_{i}.png')

    # Convertir en niveaux de gris
    gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)

    # Appliquer un filtre pour détecter les contours
    edges = cv2.Canny(gray, 50, 150)

    # Trouver les contours
    contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

    # Compter le nombre de contours détectés
    total_pins += len(contours)

print(f'Nombre total de stations-service (approximatif) : {total_pins}')


In [ ]:
############################## ola #################################

In [2]:
import re
import json
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import Select
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import time

# Spécifiez le chemin du ChromeDriver via la classe Service
service = Service('C:/Users/user/Desktop/stage/chromedriver-win64/chromedriver.exe')
driver = webdriver.Chrome(service=service)

# Accéder à la page web
driver.get("https://olaenergy.com/service-stations/")


# Attendre que les éléments de script soient présents
WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, 'script')))

# Trouver les scripts contenant les données JSON-like
scripts = driver.find_elements(By.TAG_NAME, 'script')

# Initialiser le compteur
count = 0

# Définir le modèle regex pour correspondre au motif
pattern = re.compile(r'"country":"Tunisia"')

# Parcourir les scripts et chercher des correspondances
for script in scripts:
    script_content = script.get_attribute('innerHTML')
    
    # Vérifiez si le script contient au moins "country" avant de faire correspondre le motif
    if '"country":"' in script_content:
        matches = pattern.findall(script_content)
        count += len(matches)

    # Limiter le nombre de scripts à analyser (par exemple, aux 10 premiers) pour éviter une boucle infinie
    if count > 0:
        break

print(f'Number of occurrences of the pattern: {count}')

# Fermer le navigateur
driver.quit()


Number of occurrences of the pattern: 193


In [7]:
pip install requests pandas openpyxl


Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [ ]:
######################################### shell #######################################################

In [12]:
import requests
import pandas as pd
from io import BytesIO
from bs4 import BeautifulSoup

# URL of the webpage containing the download link
page_url = 'http://catalog.industrie.gov.tn/dataset/stations-shell/resource/6627957a-f292-4b3f-8d85-0346cf71e2bc'

# Send a GET request to fetch the webpage
response = requests.get(page_url)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content of the page
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the link to the XLS file
    download_link = soup.find('a', class_='btn btn-primary resource-url-analytics resource-type-None')
    if download_link:
        file_url = download_link.get('href')
        print(f'Download link: {file_url}')
        
        # Send a GET request to download the file
        file_response = requests.get(file_url)
        
        # Check if the file request was successful
        if file_response.status_code == 200:
            # Read the file into a BytesIO object
            file_content = BytesIO(file_response.content)
            
            # Load the XLS file into a pandas DataFrame
            df = pd.read_excel(file_content)
            
            # Remove rows where either 'Lon' or 'Lat' is empty
            df = df.dropna(subset=['Lon', 'Lat'])
            
            # Remove duplicate rows based on the combination of 'Lon' and 'Lat'
            df = df.drop_duplicates(subset=['Lon', 'Lat'])
            
            # Calculate the number of remaining rows
            num_rows = df.shape[0]
            print(f'Number of rows with unique non-empty Lon and Lat: {num_rows}')
        else:
            print(f'Failed to download file. Status code: {file_response.status_code}')
    else:
        print('Download link not found.')
else:
    print(f'Failed to retrieve webpage. Status code: {response.status_code}')


Download link: http://catalog.industrie.gov.tn/dataset/5464c702-53c7-4913-bd70-d07f73acf43c/resource/6627957a-f292-4b3f-8d85-0346cf71e2bc/download/shellstations.xlsx
Number of rows with unique non-empty Lon and Lat: 167


In [ ]:
################################################# agil ####################################

In [13]:
import requests
import pandas as pd
from io import BytesIO
from bs4 import BeautifulSoup

# URL of the webpage containing the download link
page_url = 'http://catalog.industrie.gov.tn/dataset/reseaux-stations-services-agil/resource/b4f99c73-cabf-4132-81e6-9663b9954d0c'

# Send a GET request to fetch the webpage
response = requests.get(page_url)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content of the page
    soup = BeautifulSoup(response.content, 'html.parser')

    # Find the link to the XLS file
    download_link = soup.find('a', class_='btn btn-primary resource-url-analytics resource-type-None')
    if download_link:
        file_url = download_link.get('href')
        print(f'Download link: {file_url}')
        
        # Send a GET request to download the file
        file_response = requests.get(file_url)
        
        # Check if the file request was successful
        if file_response.status_code == 200:
            # Read the file into a BytesIO object
            file_content = BytesIO(file_response.content)
            
            # Load the XLS file into a pandas DataFrame
            df = pd.read_excel(file_content)
            
            # Remove rows where either 'Lon' or 'Lat' is empty
            df = df.dropna(subset=['Lon', 'Lat'])
            
            # Remove duplicate rows based on the combination of 'Lon' and 'Lat'
            df = df.drop_duplicates(subset=['Lon', 'Lat'])
            
            # Calculate the number of remaining rows
            num_rows = df.shape[0]
            print(f'Number of rows with unique non-empty Lon and Lat: {num_rows}')
        else:
            print(f'Failed to download file. Status code: {file_response.status_code}')
    else:
        print('Download link not found.')
else:
    print(f'Failed to retrieve webpage. Status code: {response.status_code}')

Download link: http://catalog.industrie.gov.tn/dataset/cac7de4d-65fb-4b16-b85a-c319f33f38e9/resource/b4f99c73-cabf-4132-81e6-9663b9954d0c/download/stationsservicesndpagil.xls
Number of rows with unique non-empty Lon and Lat: 207


In [ ]:
########################################## Total energies #################################""

In [16]:
import requests
from bs4 import BeautifulSoup

# URL of the webpage to scrape
page_url = 'https://totalenergies.com/fr/tunisie'

# Send a GET request to fetch the webpage
response = requests.get(page_url)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content of the page
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Find the div containing the keyword "stations"
    p_tag = soup.find('p', string=lambda text: text and 'stations' in text.lower())
    
    if p_tag:
        # Navigate to the parent div containing the number
        parent_div = p_tag.find_parent('div', class_='block-key-figure')
        if parent_div:
            # Find the div with the specific class containing the number
            number_div = parent_div.find('div', class_='inner-value blue_degraded')
            if number_div:
                # Extract the number from the <span> tag within the div
                number = number_div.find('span').text.strip()
                print(f'Number extracted: {number}')
            else:
                print('Div with class "inner-value blue_degraded" not found.')
        else:
            print('Parent div with class "block-key-figure" not found.')
    else:
        print('Paragraph containing the keyword "stations" not found.')
else:
    print(f'Failed to retrieve webpage. Status code: {response.status_code}')


Number extracted: 160


In [ ]:
############################### nombre total ##############################""

In [19]:
import requests
from bs4 import BeautifulSoup
import re

# URL of the webpage to scrape
page_url = 'https://www.energiemines.gov.tn/fr/themes/energie/hydrocarbures/raffinage-transport-stockage-et-distribution-du-petrole/distribution-des-carburants-et-gpl/'

# Send a GET request to fetch the webpage, ignoring SSL certificate verification
response = requests.get(page_url, verify=False)

# Check if the request was successful
if response.status_code == 200:
    # Parse the HTML content of the page
    soup = BeautifulSoup(response.content, 'html.parser')
    
    # Extract text content from the page
    text = soup.get_text(separator=' ')
    
    # Search for the keyword "stations-service" and extract numbers around it
    keyword = 'stations-service'
    # Adjust the pattern to handle various cases where numbers might be around the keyword
    pattern = rf'(\d+)\s*{keyword}|\s*{keyword}\s*(\d+)'
    matches = re.findall(pattern, text, re.IGNORECASE)
    
    # Extract and print numbers
    numbers = [match[0] or match[1] for match in matches]
    if numbers:
        print(f'Numbers found around "{keyword}": {numbers}')
    else:
        # Special handling for cases where the keyword and number are in the same sentence
        special_pattern = rf'{keyword}[^.]*?(\d+)'
        special_matches = re.findall(special_pattern, text, re.IGNORECASE)
        if special_matches:
            print(f'Numbers found in context of "{keyword}": {special_matches}')
        else:
            print(f'No numbers found around "{keyword}".')
else:
    print(f'Failed to retrieve webpage. Status code: {response.status_code}')


Numbers found around "stations-service": ['850']


C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1045: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.energiemines.gov.tn'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [25]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import re
from selenium import webdriver
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from io import BytesIO  # Import this for handling the Excel file

# Initialize DataFrame to store results
data = []

# Function to scrape data from each site
def scrape_olaenergy():
    service = Service('C:/Users/user/Desktop/stage/chromedriver-win64/chromedriver.exe')
    driver = webdriver.Chrome(service=service)
    driver.get("https://olaenergy.com/service-stations/")
    WebDriverWait(driver, 20).until(EC.presence_of_element_located((By.TAG_NAME, 'script')))
    scripts = driver.find_elements(By.TAG_NAME, 'script')
    count = 0
    pattern = re.compile(r'"country":"Tunisia"')
    for script in scripts:
        script_content = script.get_attribute('innerHTML')
        if '"country":"' in script_content:
            matches = pattern.findall(script_content)
            count += len(matches)
        if count > 0:
            break
    driver.quit()
    return count

def scrape_industrie_tn(url, company_name):
    response = requests.get(url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        download_link = soup.find('a', class_='btn btn-primary resource-url-analytics resource-type-None')
        if download_link:
            file_url = download_link.get('href')
            file_response = requests.get(file_url)
            if file_response.status_code == 200:
                file_content = BytesIO(file_response.content)
                df = pd.read_excel(file_content)
                df = df.dropna(subset=['Lon', 'Lat'])
                df = df.drop_duplicates(subset=['Lon', 'Lat'])
                num_rows = df.shape[0]
                return num_rows
    return None

def scrape_totalenergies():
    page_url = 'https://totalenergies.com/fr/tunisie'
    response = requests.get(page_url)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        p_tag = soup.find('p', string=lambda text: text and 'stations' in text.lower())
        if p_tag:
            parent_div = p_tag.find_parent('div', class_='block-key-figure')
            if parent_div:
                number_div = parent_div.find('div', class_='inner-value blue_degraded')
                if number_div:
                    number = number_div.find('span').text.strip()
                    return int(number)
    return None

def scrape_energiemines():
    page_url = 'https://www.energiemines.gov.tn/fr/themes/energie/hydrocarbures/raffinage-transport-stockage-et-distribution-du-petrole/distribution-des-carburants-et-gpl/'
    response = requests.get(page_url, verify=False)
    if response.status_code == 200:
        soup = BeautifulSoup(response.content, 'html.parser')
        text = soup.get_text(separator=' ')
        keyword = 'stations-service'
        pattern = rf'(\d+)\s*{keyword}|\s*{keyword}\s*(\d+)'
        matches = re.findall(pattern, text, re.IGNORECASE)
        numbers = [match[0] or match[1] for match in matches]
        if numbers:
            return int(numbers[0])
    return None

# Scrape data from different sources
olaenergy_stations = scrape_olaenergy()
Shell_stations = scrape_industrie_tn('http://catalog.industrie.gov.tn/dataset/stations-shell/resource/6627957a-f292-4b3f-8d85-0346cf71e2bc', 'Shell')
Agil_stations = scrape_industrie_tn('http://catalog.industrie.gov.tn/dataset/reseaux-stations-services-agil/resource/b4f99c73-cabf-4132-81e6-9663b9954d0c', 'Agil')
totalenergies_stations = scrape_totalenergies()
energiemines_stations = scrape_energiemines()

# Add the results to the DataFrame
data.append({'Company': 'Ola Energy', 'Stations': olaenergy_stations})
data.append({'Company': 'Shell', 'Stations': Shell_stations})
data.append({'Company': 'Agil', 'Stations': Agil_stations})
data.append({'Company': 'Total Energies', 'Stations': totalenergies_stations})

# Create DataFrame
df = pd.DataFrame(data)

# Calculate the total number of stations
total_stations = df['Stations'].sum()

# Calculate the difference between the total and the number of stations for each company
others = energiemines_stations - total_stations

# Add row for 'Other' category
other_df = pd.DataFrame([{'Company': 'Other', 'Stations': others}])
df = pd.concat([df, other_df], ignore_index=True)

print(df)


          Company  Stations
0      Ola Energy       193
1           Shell       167
2            Agil       207
3  Total Energies       160
4           Other       123


C:\ProgramData\Anaconda3\lib\site-packages\urllib3\connectionpool.py:1045: InsecureRequestWarning: Unverified HTTPS request is being made to host 'www.energiemines.gov.tn'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/1.26.x/advanced-usage.html#ssl-warnings
  warnings.warn(


In [27]:
import pandas as pd
import pyodbc

# SQL Server connection details
server = 'LAPTOP-HED7JE2F'
database = 'PGS_STAGE_DW'
username = 'sa'
password = 'sa@123@123'

# Establish connection
conn = pyodbc.connect(f'DRIVER={{SQL Server}};SERVER={server};DATABASE={database};UID={username};PWD={password}')
cursor = conn.cursor()

# Step 1: Delete existing data
delete_query = 'DELETE FROM nb_stations'
cursor.execute(delete_query)
print("Existing data deleted from nb_stations.")

# Step 2: Insert data from DataFrame into `nb_stations`
insert_query = '''
INSERT INTO nb_stations (nom_concurent, nombres)
VALUES (?, ?)
'''

# Iterate over DataFrame rows and insert data
for index, row in df.iterrows():
    cursor.execute(insert_query, (row['Company'], row['Stations']))

# Commit the transaction and close the connection
conn.commit()
conn.close()

print("Data successfully inserted into nb_stations.")

Existing data deleted from nb_stations.
Data successfully inserted into nb_stations.
